In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
import torch

# Load the model
model = YOLO("yolov8n.pt")

# Save the model
torch.save(model, "saved_yolo.pt")

# Load the saved model
loaded_model = torch.load("saved_yolo.pt")

# Print the model summary
print(loaded_model)

In [ ]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('data/export/_annotations.csv')

# Remove empty rows
df_cleaned = df.dropna(how='all')

# Save the cleaned data back to a CSV file
df_cleaned.to_csv('data/export/annotations.csv', index=False)

In [ ]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('data/export/annotations.csv')

# Specify the column name
column_name = 'class'

# Find unique values in the specified column
unique_values = df[column_name].unique()

# Print the unique values
print(f"Unique values in the '{column_name}' column:")
for value in unique_values:
    print(value)

# Optional: Print the count of unique values
print(f"\nTotal number of unique values: {len(unique_values)}")


In [ ]:
class_names = ['car','pedestrian','biker','truck','trafficLight-Red','trafficLight','trafficLight-Green',
               'trafficLight-RedLeft','trafficLight-GreenLeft','trafficLight-Yellow','trafficLight-YellowLeft']
class_dict = {item: 0 for item in class_names}
class_dict

In [1]:
import torch
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from transformers import SegformerFeatureExtractor, SegformerForSemanticSegmentation

In [2]:
# Load Segmentation Model (for Road Detection)
seg_feature_extractor = SegformerFeatureExtractor.from_pretrained("nvidia/segformer-b5-finetuned-cityscapes-1024-1024")
seg_model = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b5-finetuned-cityscapes-1024-1024")
# Move segmentation model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seg_model = seg_model.to(device)

E:\Supporting Tools\Python\Lib\site-packages\transformers\models\segformer\feature_extraction_segformer.py:28: FutureWarning: The class SegformerFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use SegformerImageProcessor instead.
  warnings.warn(
E:\Supporting Tools\Python\Lib\site-packages\transformers\utils\deprecation.py:172: UserWarning: The following named arguments are not valid for `SegformerFeatureExtractor.__init__` and were ignored: 'feature_extractor_type'
  return func(*args, **kwargs)


In [3]:
def get_road_mask(image_path, target_size):
    """Performs semantic segmentation to detect roads and returns a resized binary mask."""
    image = Image.open(image_path).convert("RGB")
    inputs = seg_feature_extractor(images=image, return_tensors="pt")
    # Move inputs to GPU
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Get segmentation output
    with torch.no_grad():
        outputs = seg_model(**inputs)
    logits = outputs.logits
    # Move logits back to CPU for NumPy operations
    segmentation = logits.argmax(dim=1).squeeze().cpu().numpy()
    
    # Define road class index (based on Cityscapes dataset)
    ROAD_CLASS_INDEX = 0  # Adjust this if needed
    # Create binary mask (255 for roads, 0 for others)
    road_mask = np.where(segmentation == ROAD_CLASS_INDEX, 255, 0).astype(np.uint8)
    # Resize road mask to match depth image size
    road_mask = cv2.resize(road_mask, target_size, interpolation=cv2.INTER_NEAREST)
    return Image.fromarray(road_mask)

In [4]:
img=Image.open(r"dataset\images\val\1478900957671553561_jpg.rf.GZVdKZdkyUXLyD18XyCm.jpg")
size=img.size

In [5]:
road=get_road_mask(r"dataset\images\val\1478900957671553561_jpg.rf.GZVdKZdkyUXLyD18XyCm.jpg",size)

In [6]:
road.save("road_map.png")

In [1]:
from PIL import Image, ImageDraw
import csv
import os

In [2]:
def transfer_bounding_boxes(annotations_csv, target_image_path, output_path, source_image_name):
    """
    Transfer bounding boxes from CSV annotations to a target image
    Only transfers boxes associated with the specified source image name
    
    Parameters:
    annotations_csv (str): Path to CSV file containing bounding box annotations
    target_image_path (str): Path to the target image where boxes will be drawn
    output_path (str): Path where the output image will be saved
    source_image_name (str): Name of the source image to filter annotations
    """
    # Open target image 
    target_img = Image.open(target_image_path)
    target_width, target_height = target_img.size
    print(f"Target image dimensions: {target_width}x{target_height}")
    
    # Create drawing object
    draw = ImageDraw.Draw(target_img)
    
    # Read bounding box data from CSV
    boxes_drawn = 0
    with open(annotations_csv, 'r') as csvfile:
        reader = csv.reader(csvfile)
        # Read header
        header = next(reader, None)
        print(f"CSV header: {header}")
        
        # Verify header matches expected structure
        if header != ['filename', 'class', 'confidence', 'xmin', 'ymin', 'xmax', 'ymax']:
            print(f"Warning: CSV header doesn't match expected structure. Found: {header}")
            print("Expected: ['filename', 'class', 'confidence', 'xmin', 'ymin', 'xmax', 'ymax']")
        
        # Process each row
        for row in reader:
            try:
                # Check if this row is for our target image
                if len(row) < 7:  # Ensure row has all required columns
                    print(f"Warning: Row has insufficient columns: {row}")
                    continue
                    
                filename = row[0]
                
                # Skip if not matching our specified image name
                if filename != source_image_name:
                    continue
                
                # Get class and confidence
                class_name = row[1]
                confidence = float(row[2])
                
                # Get bounding box coordinates (using xmin, ymin, xmax, ymax directly)
                xmin = int(float(row[3]))
                ymin = int(float(row[4]))
                xmax = int(float(row[5]))
                ymax = int(float(row[6]))
                
                # Draw rectangle
                draw.rectangle([xmin, ymin, xmax, ymax], outline="red", width=2)
                
                # Add label text with class and confidence
                label = f"{class_name}: {confidence:.2f}"
                draw.text((xmin, ymin - 15), label, fill="red")
                
                boxes_drawn += 1
            except (IndexError, ValueError) as e:
                print(f"Error processing row {row}: {e}")
    
    # Save the output image
    target_img.save(output_path)
    print(f"Process complete. {boxes_drawn} bounding boxes drawn for image '{source_image_name}' and saved to {output_path}")

In [4]:
# Example usage
if __name__ == "__main__":
    # Set your file paths here
    annotations_csv = "detection_results.csv"
    target_image_path = "dsimg.png"
    output_path = "combo.png"
    source_image_name = "1478900957671553561_jpg.rf.GZVdKZdkyUXLyD18XyCm.jpg"  # The image name to filter by
    
    transfer_bounding_boxes(annotations_csv, target_image_path, output_path, source_image_name)

Target image dimensions: 512x512
CSV header: ['filename', 'class', 'confidence', 'xmin', 'ymin', 'xmax', 'ymax']
Process complete. 9 bounding boxes drawn for image '1478900957671553561_jpg.rf.GZVdKZdkyUXLyD18XyCm.jpg' and saved to combo.png


In [1]:
import pandas as pd

# Load the CSV file
filename = 'data/export/annotations.csv'  # Replace with the actual CSV file name
csv_data = pd.read_csv(filename)

# Define the target classes
target_classes = ['car', 'pedestrian', 'biker', 'truck']

# Specify the filename you are interested in
specific_filename = '1478900957671553561_jpg.rf.GZVdKZdkyUXLyD18XyCm.jpg'  # Replace with the filename you want to process

# Filter rows for the specific filename and target classes
filtered_data = csv_data[(csv_data['filename'] == specific_filename) & (csv_data['class'].isin(target_classes))]

# Extract tuples of (xmin, ymin, xmax, ymax)
bbox_tuples = list(zip(filtered_data['xmin'], filtered_data['ymin'], filtered_data['xmax'], filtered_data['ymax']))

# Print or use the results as needed
print(f"Bounding boxes for {specific_filename}: {bbox_tuples}")


Bounding boxes for 1478900957671553561_jpg.rf.GZVdKZdkyUXLyD18XyCm.jpg: [(503, 250, 512, 310), (19, 256, 57, 290), (65, 259, 104, 292), (108, 259, 138, 285), (110, 258, 136, 282), (140, 253, 176, 289), (145, 259, 174, 288), (187, 233, 281, 361), (306, 151, 399, 317)]


In [2]:
import cv2
import numpy as np

def estimate_distances(depth_image_path, bounding_boxes):
    # Load the depth image (16-bit PNG)
    depth_image = cv2.imread(depth_image_path, cv2.IMREAD_UNCHANGED)

    if depth_image is None:
        raise ValueError("Failed to load depth image. Check the file path.")

    distances = []
    for bbox in bounding_boxes:
        xmin, ymin, xmax, ymax = bbox

        # Extract the region of interest (ROI) from depth image
        roi_depth = depth_image[ymin:ymax, xmin:xmax]

        # Calculate estimated distance (e.g., median depth to avoid noise)
        if roi_depth.size > 0:
            estimated_distance = np.median(roi_depth[roi_depth > 0])  # Ignore zero-depth values
            distances.append(estimated_distance)
        else:
            distances.append(float('inf'))  # Handle cases where ROI is empty

    return distances

# Example usage:
depth_image_path = "data/export/depth/depth_1478900957671553561_jpg.rf.GZVdKZdkyUXLyD18XyCm.png"  # Provide your depth image file path
bounding_boxes = [(503, 250, 512, 310), (19, 256, 57, 290), (65, 259, 104, 292), (108, 259, 138, 285), (110, 258, 136, 282), 
                  (140, 253, 176, 289), (145, 259, 174, 288), (187, 233, 281, 361), (306, 151, 399, 317)]  # Example bounding boxes

estimated_distances = estimate_distances(depth_image_path, bounding_boxes)
print("Estimated distances:", estimated_distances)


Estimated distances: [104.0, 116.0, 87.0, 81.0, 80.0, 80.0, 80.0, 140.0, 97.0]


In [3]:
import os
import pandas as pd

# Specify the folder path
folder_path = 'data/export/'  # Replace with the actual folder path

try:
    # List all JPG files in the folder
    jpg_files = [file for file in os.listdir(folder_path) if file.endswith('.jpg')]

    # Create a DataFrame with filenames
    df = pd.DataFrame({'filename': jpg_files})

    # Save the DataFrame to a CSV file
    output_csv = 'full.csv'
    df.to_csv(output_csv, index=False)

    print(f"CSV file '{output_csv}' created successfully with {len(jpg_files)} entries.")
except FileNotFoundError:
    print(f"Error: The folder '{folder_path}' does not exist. Please provide a valid path.")


CSV file 'full.csv' created successfully with 29801 entries.


In [5]:
import os
import pandas as pd

# Specify the folder path
folder_path = 'data/export/'  # Replace with the actual folder path

# Dictionary to store unique files based on their prefix
unique_files = {}

try:
    # List all JPG files in the folder
    jpg_files = [file for file in os.listdir(folder_path) if file.endswith('.jpg')]

    # Iterate through each file and keep only one copy per prefix
    for file in jpg_files:
        # Extract the prefix before the first '.' (e.g., '1478900957671553561_jpg')
        prefix = file.split('.')[0]

        # If this prefix is not already in the dictionary, add it
        if prefix not in unique_files:
            unique_files[prefix] = file

    # Get the list of unique filenames
    filtered_files = list(unique_files.values())

    # Create a DataFrame with the filtered filenames
    df = pd.DataFrame({'filename': filtered_files})

    # Save the DataFrame to a CSV file named 'filtered.csv'
    output_csv = 'filtered.csv'
    df.to_csv(output_csv, index=False)

    print(f"CSV file '{output_csv}' created successfully with {len(filtered_files)} entries.")
except FileNotFoundError:
    print(f"Error: The folder '{folder_path}' does not exist. Please provide a valid path.")


CSV file 'filtered.csv' created successfully with 15000 entries.


In [6]:
import pandas as pd

try:
    # Load both CSV files
    annotations_df = pd.read_csv('data/export/annotations.csv')
    filtered_df = pd.read_csv('filtered.csv')
    
    # Filter annotations to keep only filenames present in filtered.csv
    filtered_annotations = annotations_df[annotations_df['filename'].isin(filtered_df['filename'])]
    
    # Save the filtered results
    filtered_annotations.to_csv('filtered_annotations.csv', index=False)
    print(f"Filtered annotations saved to filtered_annotations.csv ({len(filtered_annotations)} entries kept)")
    
except FileNotFoundError as e:
    print(f"Error: {e.filename} not found. Please verify:")
    print("- annotations.csv exists in current directory")
    print("- filtered.csv exists in current directory")
    print("- Both files contain a 'filename' column")


Filtered annotations saved to filtered_annotations.csv (97942 entries kept)


In [5]:
import pandas as pd

try:
    # Load the filtered.csv and objDepth.csv files
    filtered_df = pd.read_csv('filtered.csv')
    objDepth_df = pd.read_csv('data/export/objDepth.csv')

    # Ensure both files have the 'filename' column
    if 'filename' not in filtered_df.columns or 'filename' not in objDepth_df.columns:
        raise KeyError("Both CSV files must contain a 'filename' column.")

    # Add the prefix "data/export/" to filenames in filtered.csv for matching
    filtered_filenames_with_prefix = "data/export/" + filtered_df['filename']

    # Filter rows in objDepth.csv where filename matches the prefixed filenames
    filtered_objDepth = objDepth_df[objDepth_df['filename'].isin(filtered_filenames_with_prefix)]

    # Save the filtered data to a new CSV file
    output_csv = 'filtered_objDepth.csv'
    filtered_objDepth.to_csv(output_csv, index=False)

    print(f"Filtered data saved to {output_csv} ({len(filtered_objDepth)} rows).")

except FileNotFoundError as e:
    print(f"Error: {e.filename} not found. Please ensure the file exists.")
except KeyError as e:
    print(f"Error: {e}.")


Filtered data saved to filtered_objDepth.csv (15000 rows).
